In [ ]:
# 1) GPU check + package installs
import importlib
import sys

print(sys.version)

!nvidia-smi
!{sys.executable} -m pip install -q --upgrade pip
!{sys.executable} -m pip install -q pretty_midi mido tqdm numpy

# Verify packages are importable
for module_name in ['pretty_midi', 'mido', 'tqdm', 'numpy']:
    importlib.import_module(module_name)
print('Package check passed: pretty_midi, mido, tqdm, numpy')

In [ ]:
# pyright: reportMissingImports=false, reportMissingModuleSource=false
# 2) All imports + baseline config
import csv
import random
import shutil
import sys
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pretty_midi

SEED = 42
OUTPUT_ROOT = Path('/kaggle/working/outputs')
GENERATED_MIDI_DIR = OUTPUT_ROOT / 'generated_midis'
PLOTS_DIR = OUTPUT_ROOT / 'plots'

for directory in [OUTPUT_ROOT, GENERATED_MIDI_DIR, PLOTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)

set_seed(SEED)
print('Device: CPU (baselines)')
print('Output dir:', GENERATED_MIDI_DIR)

In [ ]:
# 3) Data loading - MAESTRO dataset exploration
DATA_INPUT_CANDIDATES = [
    Path('/kaggle/input/datasets/jackvial/themaestrodatasetv2'),
    Path('/kaggle/input/themaestrodatasetv2'),
    Path('/kaggle/input/maestro-v3.0.0'),
    Path('/kaggle/input/maestro'),
]

data_root = None
for candidate in DATA_INPUT_CANDIDATES:
    if candidate.exists():
        data_root = candidate
        break

if data_root is None:
    raise FileNotFoundError(
        'MAESTRO dataset not found. Run "!ls /kaggle/input" to find the correct path.'
    )

midi_files = list(data_root.glob('**/*.midi')) + list(data_root.glob('**/*.mid'))
midi_files = midi_files[:50]  # Sample first 50 for baseline training
print(f'Found {len(midi_files)} MIDI files in {data_root}')
print(f'Using {len(midi_files)} files for baseline training')

In [ ]:
# 4) Extract note sequences from MIDI files for baselines
def extract_note_sequence(midi_path: Path) -> list:
    """Extract a sequence of MIDI note pitches from a file."""
    try:
        pm = pretty_midi.PrettyMIDI(str(midi_path))
        notes = []
        for instrument in pm.instruments:
            if instrument.is_drum:
                continue
            for note in instrument.notes:
                notes.append(int(note.pitch))
        return sorted(notes) if notes else []
    except Exception as e:
        print(f'Error loading {midi_path}: {e}')
        return []

all_note_sequences = []
all_notes = []
for midi_file in midi_files:
    seq = extract_note_sequence(midi_file)
    if len(seq) > 10:
        all_note_sequences.append(seq)
        all_notes.extend(seq)

print(f'Extracted {len(all_note_sequences)} sequences')
print(f'Total note events: {len(all_notes)}')
print(f'Min pitch: {min(all_notes) if all_notes else 0}')
print(f'Max pitch: {max(all_notes) if all_notes else 127}')

In [ ]:
# 5) Random baseline - uniform random note selection
def random_baseline_generator(num_samples: int = 10, sequence_length: int = 100) -> list:
    """Generate random note sequences by uniform sampling from observed pitches."""
    generated = []
    for _ in range(num_samples):
        sequence = np.random.choice(all_notes, size=sequence_length)
        generated.append(sequence.tolist())
    return generated

random_sequences = random_baseline_generator(num_samples=10)
print(f'Generated {len(random_sequences)} random sequences')

In [ ]:
# 6) Markov chain baseline - first-order transitions
def build_markov_transition_matrix(sequences: list) -> dict:
    """Build first-order Markov transition matrix from note sequences."""
    transitions = defaultdict(lambda: defaultdict(int))
    
    for seq in sequences:
        for i in range(len(seq) - 1):
            current = seq[i]
            next_note = seq[i + 1]
            transitions[current][next_note] += 1
    
    # Convert counts to probabilities
    probs = {}
    for state, next_states in transitions.items():
        total = sum(next_states.values())
        probs[state] = {
            next_state: count / total
            for next_state, count in next_states.items()
        }
    
    return probs

markov_chain = build_markov_transition_matrix(all_note_sequences)
print(f'Built Markov chain with {len(markov_chain)} states')

In [ ]:
# 7) Markov chain generation
def markov_baseline_generator(
    markov_chain: dict,
    start_note: int = 60,
    num_samples: int = 10,
    sequence_length: int = 100,
) -> list:
    """Generate sequences using first-order Markov chain."""
    generated = []
    
    for _ in range(num_samples):
        sequence = [start_note]
        current = start_note
        
        for _ in range(sequence_length - 1):
            if current in markov_chain:
                next_states = markov_chain[current]
                notes = list(next_states.keys())
                probs = list(next_states.values())
                current = np.random.choice(notes, p=probs)
            else:
                current = np.random.choice(all_notes)
            sequence.append(current)
        
        generated.append(sequence)
    
    return generated

markov_sequences = markov_baseline_generator(markov_chain, num_samples=10)
print(f'Generated {len(markov_sequences)} Markov sequences')

In [ ]:
# 8) Convert note sequences to MIDI files
def notes_to_midi(note_sequence: list, output_path: Path, tempo_bpm: float = 120.0) -> None:
    """Convert a sequence of MIDI pitches to a MIDI file."""
    pm = pretty_midi.PrettyMIDI()
    instrument = pretty_midi.Instrument(program=0, is_drum=False)
    pm.instruments.append(instrument)
    
    # Convert notes to note events with timing
    current_time = 0.0
    note_duration = 60.0 / tempo_bpm / 4  # 16th note duration
    
    for pitch in note_sequence:
        note = pretty_midi.Note(
            velocity=64,
            pitch=int(np.clip(pitch, 21, 108)),
            start=current_time,
            end=current_time + note_duration,
        )
        instrument.notes.append(note)
        current_time += note_duration
    
    pm.write(str(output_path))

# Generate MIDI files
random_paths = []
for i, seq in enumerate(random_sequences):
    path = GENERATED_MIDI_DIR / f'baseline_random_{i+1}.mid'
    notes_to_midi(seq, path)
    random_paths.append(path)
    print(f'Saved: {path}')

markov_paths = []
for i, seq in enumerate(markov_sequences):
    path = GENERATED_MIDI_DIR / f'baseline_markov_{i+1}.mid'
    notes_to_midi(seq, path)
    markov_paths.append(path)
    print(f'Saved: {path}')

In [ ]:
# 9) Evaluation - compute same metrics as Tasks 1-4
def pitch_histogram_distance(seq1: list, seq2: list) -> float:
    """Compute L1 distance between pitch histograms (0-11 chromatic scale)."""
    hist1 = np.zeros(12)
    hist2 = np.zeros(12)
    
    for pitch in seq1:
        hist1[int(pitch) % 12] += 1
    for pitch in seq2:
        hist2[int(pitch) % 12] += 1
    
    hist1 = hist1 / (np.sum(hist1) + 1e-6)
    hist2 = hist2 / (np.sum(hist2) + 1e-6)
    
    return float(np.sum(np.abs(hist1 - hist2)))

def rhythm_diversity(seq: list) -> float:
    """Compute rhythm diversity as ratio of unique intervals."""
    if len(seq) < 2:
        return 0.0
    intervals = [abs(seq[i+1] - seq[i]) for i in range(len(seq)-1)]
    unique = len(set(intervals))
    return float(unique / len(intervals))

def compute_baseline_metrics(sequences: list, baseline_type: str) -> dict:
    """Compute metrics for a set of generated sequences."""
    pitch_diversities = []
    rhythm_scores = []
    
    for seq in sequences:
        pitch_diversities.append(rhythm_diversity(seq))
        rhythm_scores.append(rhythm_diversity(seq))
    
    return {
        'type': baseline_type,
        'mean_pitch_diversity': float(np.mean(pitch_diversities)) if pitch_diversities else 0.0,
        'mean_rhythm_diversity': float(np.mean(rhythm_scores)) if rhythm_scores else 0.0,
    }

random_metrics = compute_baseline_metrics(random_sequences, 'Random')
markov_metrics = compute_baseline_metrics(markov_sequences, 'Markov')

print('\\nBaseline Evaluation Metrics:')
print(f"Random:  Pitch Div={random_metrics['mean_pitch_diversity']:.3f}, Rhythm Div={random_metrics['mean_rhythm_diversity']:.3f}")
print(f"Markov:  Pitch Div={markov_metrics['mean_pitch_diversity']:.3f}, Rhythm Div={markov_metrics['mean_rhythm_diversity']:.3f}")

In [ ]:
# 10) Save baseline results to CSV for comparison
baseline_results_csv = OUTPUT_ROOT / 'baseline_results.csv'
with baseline_results_csv.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['baseline_type', 'mean_pitch_diversity', 'mean_rhythm_diversity'])
    writer.writeheader()
    writer.writerow(random_metrics)
    writer.writerow(markov_metrics)

print(f'Baseline results saved to {baseline_results_csv}')

In [ ]:
# 11) Summary - deliverables check
expected_random = [GENERATED_MIDI_DIR / f'baseline_random_{i}.mid' for i in range(1, 11)]
expected_markov = [GENERATED_MIDI_DIR / f'baseline_markov_{i}.mid' for i in range(1, 11)]

print('Baseline Deliverables Summary')
print('\\nRandom Generator:')
for path in expected_random:
    status = 'OK' if path.exists() else 'MISSING'
    print(f'[{status}] {path.name}')

print('\\nMarkov Chain Generator:')
for path in expected_markov:
    status = 'OK' if path.exists() else 'MISSING'
    print(f'[{status}] {path.name}')

print(f'\\nBaseline results: {baseline_results_csv}')